In [1]:
from datasets import load_dataset

# Load the dataset from the hub
dataset = load_dataset("TheFinAI/fiqa-sentiment-classification")

Intel MKL WARNING: Support of Intel(R) Streaming SIMD Extensions 4.2 (Intel(R) SSE4.2) enabled only processors has been deprecated. Intel oneAPI Math Kernel Library 2025.0 will require Intel(R) Advanced Vector Extensions (Intel(R) AVX) instructions.
Intel MKL WARNING: Support of Intel(R) Streaming SIMD Extensions 4.2 (Intel(R) SSE4.2) enabled only processors has been deprecated. Intel oneAPI Math Kernel Library 2025.0 will require Intel(R) Advanced Vector Extensions (Intel(R) AVX) instructions.


In [2]:
# Dataset shape, total number of rows: 1173
dataset.shape

{'train': (822, 6), 'test': (234, 6), 'valid': (117, 6)}

In [3]:
import pandas as pd
import numpy as np

train = dataset['train'].to_pandas()
test = dataset['test'].to_pandas()
val = dataset['valid'].to_pandas()
data = pd.concat([train, test, val], ignore_index=True)

In [4]:
event_map = {
    # Financial Affairs
    'Earnings': ('Financial Affairs', 'Profit Announcement'),
    'Earnings Beat': ('Financial Affairs', 'Profit Announcement'),
    'Earnings miss': ('Financial Affairs', 'Profit Announcement'),
    'Financial Results': ('Financial Affairs', 'Profit Announcement'),
    'Profit Warning': ('Financial Affairs', 'Profit Announcement'),
    'Company Guidance': ('Financial Affairs', 'Profit Forecast'),
    'Future Price': ('Financial Affairs', 'Profit Forecast'),
    'Financial': ('Financial Affairs', 'Other Financial Affairs'),
    'Cash Flow': ('Financial Affairs', 'Other Financial Affairs'),
    'Cash Reserves': ('Financial Affairs', 'Other Financial Affairs'),
    'Fundamentals': ('Financial Affairs', 'Other Financial Affairs'),
    'Accounting': ('Financial Affairs', 'Other Financial Affairs'),
    'Sales': ('Financial Affairs', 'Other Financial Affairs'),
    'Credit Rating': ('Financial Affairs', 'Other Financial Affairs'),
    'Entering bankruptcy': ('Financial Affairs', 'Other Financial Affairs'),
    'Exisiting bankruptcy': ('Financial Affairs', 'Other Financial Affairs'),

    # Shareholder Affairs
    'Insider Trading': ('Shareholder Affairs', 'Stock Holding Adjustment'),
    'Insider Activity': ('Shareholder Affairs', 'Stock Holding Adjustment'),
    'Insider Selling': ('Shareholder Affairs', 'Stock Holding Adjustment'),
    'Stock Buyside': ('Shareholder Affairs', 'Stock Holding Adjustment'),
    'Dividend': ('Shareholder Affairs', 'Other Shareholder Affairs'),
    'Dividend going up': ('Shareholder Affairs', 'Other Shareholder Affairs'),
    'Dividend Policy': ('Shareholder Affairs', 'Other Shareholder Affairs'),

    # Stock Affairs
    'Stock Price': ('Stock Affairs', 'Stock Price Movement'),
    'Price Action': ('Stock Affairs', 'Stock Price Movement'),
    'Volatility': ('Stock Affairs', 'Stock Price Movement'),
    'Stock Volatility': ('Stock Affairs', 'Stock Price Movement'),
    'Volatilit': ('Stock Affairs', 'Stock Price Movement'),
    'Buy Signal': ('Stock Affairs', 'Stock Price Movement'),
    'Sell Signal': ('Stock Affairs', 'Stock Price Movement'),
    'Sell-Off': ('Stock Affairs', 'Stock Price Movement'),
    'Selling Pressure': ('Stock Affairs', 'Stock Price Movement'),
    'Technical Analysis': ('Stock Affairs', 'Stock Price Movement'),
    'MACD': ('Stock Affairs', 'Stock Price Movement'),
    'Stochastic': ('Stock Affairs', 'Stock Price Movement'),
    'Breakout': ('Stock Affairs', 'Stock Price Movement'),
    'Overbought': ('Stock Affairs', 'Stock Price Movement'),
    'Oversold': ('Stock Affairs', 'Stock Price Movement'),
    '52-Week High': ('Stock Affairs', 'Stock Price Movement'),
    'Trending up': ('Stock Affairs', 'Stock Price Movement'),
    'Market Trend': ('Stock Affairs', 'Stock Price Movement'),
    'Trend': ('Stock Affairs', 'Stock Price Movement'),
    'Bullish': ('Stock Affairs', 'Stock Price Movement'),
    'Bearish': ('Stock Affairs', 'Stock Price Movement'),
    'Bullish Behaviour': ('Stock Affairs', 'Stock Price Movement'),
    'Bearish Behaviour': ('Stock Affairs', 'Stock Price Movement'),
    'Share Buyback': ('Stock Affairs', 'Stock Buyback'),
    'IPO': ('Stock Affairs', 'Stock Status'),
    'Upgrade': ('Stock Affairs', 'Other Stock Affairs'),
    'Downgrade': ('Stock Affairs', 'Other Stock Affairs'),
    'AnalystRatings': ('Stock Affairs', 'Other Stock Affairs'),
    'Coverage': ('Stock Affairs', 'Other Stock Affairs'),
    'Stock Coverage': ('Stock Affairs', 'Other Stock Affairs'),
    'Short Selling': ('Stock Affairs', 'Other Stock Affairs'),
    'Short Interest Rate': ('Stock Affairs', 'Other Stock Affairs'),
    'Options': ('Stock Affairs', 'Other Stock Affairs'),
    'Bull Call Spread': ('Stock Affairs', 'Other Stock Affairs'),
    'Bear Put Spread': ('Stock Affairs', 'Other Stock Affairs'),
    'Unusual Call Activity': ('Stock Affairs', 'Other Stock Affairs'),
    'Unusual Put Activity': ('Stock Affairs', 'Other Stock Affairs'),
    'Unusually high volume': ('Stock Affairs', 'Other Stock Affairs'),
    'Unusual high volume': ('Stock Affairs', 'Other Stock Affairs'),
    'Unusual Low Volume': ('Stock Affairs', 'Other Stock Affairs'),
    'Currency': ('Stock Affairs', 'Other Stock Affairs'),

    # Business Operations
    'New Product': ('Business Operations', 'Product Dynamics'),
    'Product Flaws': ('Business Operations', 'Product Dynamics'),
    'Product Recall': ('Business Operations', 'Product Dynamics'),
    'Corporate Expansion': ('Business Operations', 'Capacity Changes'),
    'Partnership': ('Business Operations', 'Initiating Cooperation'),
    'Joint Venture': ('Business Operations', 'Initiating Cooperation'),
    'Deal': ('Business Operations', 'Initiating Cooperation'),
    'Company Agreement': ('Business Operations', 'Initiating Cooperation'),
    'Market': ('Business Operations', 'Sales, Market Share Changes'),
    'Market Outlook': ('Business Operations', 'Sales, Market Share Changes'),
    'Failed Contract Discussion': ('Business Operations', 'Sales, Market Share Changes'),
    'Demand Shortfall': ('Business Operations', 'Sales, Market Share Changes'),
    'Strategy': ('Business Operations', 'Other Business Operations Affairs'),
    'Strategies': ('Business Operations', 'Other Business Operations Affairs'),
    'Business Model': ('Business Operations', 'Other Business Operations Affairs'),
    'Business Outlook': ('Business Operations', 'Other Business Operations Affairs'),
    'Corporate Planning': ('Business Operations', 'Other Business Operations Affairs'),
    'Contract Withdrawn': ('Business Operations', 'Other Business Operations Affairs'),
    'Halt in Services': ('Business Operations', 'Other Business Operations Affairs'),
    'Sale Halt': ('Business Operations', 'Other Business Operations Affairs'),
    'Company Communication': ('Business Operations', 'Other Business Operations Affairs'),
    'Scoop': ('Business Operations', 'Other Business Operations Affairs'),
    'Rumors': ('Business Operations', 'Other Business Operations Affairs'),
    'Direction': ('Business Operations', 'Other Business Operations Affairs'),
    'Conditions': ('Business Operations', 'Other Business Operations Affairs'),
    'Reputation': ('Business Operations', 'Other Business Operations Affairs'),
    'Dispute': ('Business Operations', 'Other Business Operations Affairs'),
    'Price Competition': ('Business Operations', 'Other Business Operations Affairs'),
    'Hacks': ('Business Operations', 'Other Business Operations Affairs'),
    'Trade': ('Business Operations', 'Other Business Operations Affairs'),

    # Compliance and Credit
    'Legal': ('Compliance and Credit', 'Legal Affairs'),
    'Court Ruling': ('Compliance and Credit', 'Company Litigation'),
    'Lawsuit': ('Compliance and Credit', 'Company Litigation'),
    'Settlement': ('Compliance and Credit', 'Company Litigation'),
    'Regulatory': ('Compliance and Credit', 'Regulatory Inquiries'),
    'Risks': ('Compliance and Credit', 'Other Compliance and Credit Affairs'),
    'Sanctions': ('Compliance and Credit', 'Other Compliance and Credit Affairs'),

    # Management Affairs
    'Appointment': ('Management Affairs', 'Directors, Supervisors, and Senior Executives Dynamics'),
    'Executive Appointment': ('Management Affairs', 'Directors, Supervisors, and Senior Executives Dynamics'),
    'Executive Reshuffle': ('Management Affairs', 'Directors, Supervisors, and Senior Executives Dynamics'),
    'Staff Hiring': ('Management Affairs', 'Employee Dynamics'),
    'Reorganization': ('Management Affairs', 'Employee Dynamics'),
    'Extraordinary Meeting': ('Management Affairs', 'Directors, Supervisors, and Senior Executives Dynamics'),

    # Financing and Investment
    'M&A': ('Financing and Investment', 'Mergers and Acquisitions'),
    'Buyout': ('Financing and Investment', 'Mergers and Acquisitions'),
    'Proposed Merger': ('Financing and Investment', 'Mergers and Acquisitions'),
    'Investment': ('Financing and Investment', 'Investment Events'),
    'Divestment': ('Financing and Investment', 'Investment Events'),
    'Central Banks': ('Financing and Investment', 'Capital Flows'),
    'Monetary Policy - SNB': ('Financing and Investment', 'Capital Flows'),
    'Mutual Fund flows': ('Financing and Investment', 'Capital Flows'),
}

In [ ]:
def extract_final_aspect(aspect_path):
    """
    Extract the most specific aspect from a hierarchical path.
    For example: 'Corporate/Risks/Product Recall' -> 'Product Recall'
    """
    # Split by '/' and get the last non-empty part
    parts = [part.strip() for part in aspect_path.split('/') if part.strip()]

    return parts[-1]

In [ ]:
def map_aspect_to_event(aspect_path, event_map):
    """
    Map an aspect path to coarse and fine-grained events using the event_map.
    """
    final_aspect = extract_final_aspect(aspect_path)

    # Direct mapping
    if final_aspect in event_map:
        return event_map[final_aspect]

    # Handle special cases based on the hierarchical path
    parts = [part.strip() for part in aspect_path.split('/') if part.strip()]

    # Handle specific patterns
    if len(parts) >= 2:
        if parts[0] == 'Corporate':
            if parts[1] == 'Risks':
                return ('Compliance and Credit', 'Other Compliance and Credit Affairs')
            elif parts[1] == 'Financial':
                return ('Financial Affairs', 'Other Financial Affairs')
            elif parts[1] == 'Sales':
                return ('Business Operations', 'Sales, Market Share Changes')
            elif parts[1] == 'Strategy':
                return ('Business Operations', 'Other Business Operations Affairs')
            elif parts[1] == 'Reputation':
                return ('Business Operations', 'Other Business Operations Affairs')
            elif parts[1] == 'M&A':
                return ('Financing and Investment', 'Mergers and Acquisitions')
            elif parts[1] == 'Dividend Policy':
                return ('Shareholder Affairs', 'Other Shareholder Affairs')
        elif parts[0] == 'Stock':
            if parts[1] == 'Technical Analysis':
                return ('Stock Affairs', 'Stock Price Movement')
            elif parts[1] == 'Coverage':
                return ('Stock Affairs', 'Other Stock Affairs')
            elif parts[1] == 'Price Action':
                return ('Stock Affairs', 'Stock Price Movement')
        elif parts[0] == 'Market':
            return ('Stock Affairs', 'Other Stock Affairs')
        elif parts[0] == 'Economy':
            return ('Business Operations', 'Other Business Operations Affairs')

    # Default fallback
    return ('Other Affairs', 'Other Events')

In [ ]:
def clean_df(df):
    df = df.rename(columns={'sentence': 'Sentence', 'type': 'Type',
                   'target': 'Company', 'score': 'Sentiment'})

    # Sentment is NEU if score == 0
    # Sentiment is POS if score > 0
    # Sentiment is NEG if score < 0
    df['Sentiment'] = np.where(df['Sentiment'] == 0, 'NEU',
                               np.where(df['Sentiment'] > 0, 'POS', 'NEG'))

    # Map aspect to coarse and fine-grained events
    mapped_events = df['aspect'].apply(
        lambda x: map_aspect_to_event(x, event_map))
    df['Coarse-Grained Event'] = mapped_events.apply(lambda x: x[0])
    df['Fine-Grained Event'] = mapped_events.apply(lambda x: x[1])
    df = df.drop(columns=['aspect'], axis=1)

    df['Fine-Grained Event'] = np.where(df['Fine-Grained Event'] == "Bearish Behavior", 'Bearish',
                                        np.where(df['Fine-Grained Event'] == "Bear Position", 'Bearish',
                                        np.where(df['Fine-Grained Event'] == "Bullish Behavior", 'Bullish',
                                                 np.where(df['Fine-Grained Event'] == "Bull Position", 'Bullish',
                                                 df['Fine-Grained Event']))))

    # Clean typos and inconsistencies in the 'Company' column
    df['Company'] = df['Company'].str.replace('Lemann', 'Diageo', regex=False)
    df['Company'] = df['Company'].str.replace(
        'Coca-Cola Companp', 'Coca-Cola Company', regex=False)
    df['Company'] = df['Company'].str.replace(
        'JV Partner', 'Hammerson', regex=False)
    df['Company'] = df['Company'].str.replace(
        'Deutsche BÃ¶rse', 'Deutsche Börse', regex=False)

    return df

In [8]:
data = clean_df(data)
data.head()

,_id,Sentence,Company,Sentiment,Type,Coarse-Grained Event,Fine-Grained Event
0,1,Royal Mail chairman Donald Brydon set to step ...,Royal Mail,NEG,headline,Management Affairs,"Directors, Supervisors, and Senior Executives ..."
1,100,Slump in Weir leads FTSE down from record high,Weir,NEG,headline,Stock Affairs,Stock Price Movement
2,1000,AstraZeneca wins FDA approval for key new lung...,AstraZeneca,POS,headline,Compliance and Credit,Regulatory Inquiries
3,1002,UPDATE 1-Lloyds to cut 945 jobs as part of 3-y...,Lloyds,NEG,headline,Business Operations,Other Business Operations Affairs
4,1005,Standard Chartered Shifts Emerging-Markets Str...,Standard Chartered,NEG,headline,Business Operations,Other Business Operations Affairs


In [9]:
company_to_ticker = {
    'Royal Mail': 'RMG.L',
    'Weir': 'WEIR.L',
    'AstraZeneca': 'AZN.L',
    'Lloyds': 'LLOY.L',
    'Standard Chartered': 'STAN.L',
    'Severn Trent': 'SVT.L',
    'Glencore': 'GLEN.L',
    'Perrigo': 'PRGO',
    'Shire': 'SHP.L',
    'Barclays': 'BARC.L',
    'easyJet': 'EZJ.L',
    'Compass Group': 'CPG.L',
    'Bunzl': 'BNZL.L',
    'Aviva': 'AV.L',
    'Direct Line Insurance Group PLC': 'DLG.L',
    'Morrisons': 'MRW.L',
    'Tesco': 'TSCO.L',
    'Sainsbury': 'SBRY.L',
    'Shell': 'RDSA.L',
    'Statoil': 'EQNR',
    'Julius Baer': 'BAER.SW',
    'Dixons Carphone': 'DC.L',
    'Comcast': 'CMCSA',
    'Takeda': '4502.T',
    'Novartis': 'NVS',
    'L&G': 'LGEN.L',
    'Meggitt': 'MGGT.L',
    'Kingfisher': 'KGF.L',
    'Ashtead': 'AHT.L',
    'Baxalta': 'BXLT',
    'Sage': 'SGE.L',
    'Greene King': 'GNK.L',
    'ConAgra': 'CAG',
    'InterContinental Hotels Group': 'IHG.L',
    'BHP Billiton': 'BHP',
    'Centrica PLC': 'CNA.L',
    'John Wood Group PLC': 'WG.L',
    'Legal & General Group Plc': 'LGEN.L',
    'GlaxoSmithKline': 'GSK.L',
    'Rio Tinto': 'RIO',
    'JP Morgan Chase': 'JPM',
    'JPMorgan': 'JPM',
    'Schroders': 'SDR.L',
    'EasyJet': 'EZJ.L',
    'Balfour Beatty plc': 'BBY.L',
    'National Grid plc': 'NG.L',
    'Intertek': 'ITRK.L',
    'Sanofi': 'SNY',
    'London Stock Exchange': 'LSE.L',
    'Tullow Oil': 'TLW.L',
    'Wolseley': 'WOS.L',
    'Prudential': 'PRU.L',
    'Royal Dutch Shell': 'RDSA.L',
    'Saudi Aramco': '2222.SR',
    'CaixaBank': 'CABK.MC',
    'StanChart': 'STAN.L',
    'Tata Steel': 'TATASTEEL.NS',
    'Berkshire Hathaway': 'BRK-B',
    'Hargreaves Lansdown': 'HL.L',
    'AB InBev': 'BUD',
    'Whitbread': 'WTB.L',
    'InterContinental': 'IHG.L',
    'Det Norske': 'DETN.OL',
    'Johnson Matthey': 'JMAT.L',
    'Goldman Sachs': 'GS',
    'Asahi': '2502.T',
    'Deutsche Börse': 'DB1.DE',
    "Domino's Pizza Group plc": 'DOM.L',
    'ASOS plc': 'ASC.L',
    'Priceline': 'BKNG',
    'Coca-Cola Company': 'KO',
    'Coca-Cola FEMSA': 'KOF',
    'Fresnillo': 'FRES.L',
    'Dunelm Group plc': 'DNLM.L',
    'Great Portland Estates plc': 'GPOR.L',
    'Wells Fargo': 'WFC',
    'M&G': 'MNG.L',
    'Persimmon': 'PSN.L',
    'Ryanair': 'RYA.L',
    'Lafarge Holcim': 'HCML.SW',
    'Exxon': 'XOM',
    'Deere': 'DE',
    'Rolls-Royce': 'RR.L',
    'Auto Trader': 'AUTO.L',
    'Credit Suisse': 'CS',
    'Tesco PLC': 'TSCO.L',
    'OneMain': 'OMF',
    'Springleaf': None,
    'Tullow': 'TLW.L',
    'Kraft': 'KHC',
    'Daiichi Sankyo': '4568.T',
    'Travis Perkins': 'TPK.L',
    'Diageo': 'DGE.L',
    'Pearson': 'PSON.L',
    'TalkTalk': 'TALK.L',
    'Unilever': 'ULVR.L',
    'Imperial Tobacco': 'IMT.L',
    'Sports Direct': 'SPD.L',
    'Eli Lilly & Co': 'LLY',
    'Old Mutual': 'OML.L',
    'Halliburton': 'HAL',
    'Philip Morris': 'PM',
    'Centrica': 'CNA.L',
    'Serco': 'SRP.L',
    'Land Securities': 'LAND.L',
    'Barratt': 'BDEV.L',
    'Mondi': 'MNDI.L',
    'Gazprom': 'OGZPY',
    'Aberdeen AM': 'ADG.L',
    'Teva': 'TEVA',
    'Sophos': 'SOPH.L',
    'easyjet': 'EZJ.L',
    'Petrofac': 'PFC.L',
    'Starwood': 'HOT',
    'Hikma': 'HIK.L',
    'Ocwen': 'OCN',
    'British American Tobacco': 'BATS.L',
    'Zurich Insurance': 'ZURN.SW',
    'BAE Systems': 'BA.L',
    'Glaxo': 'GSK.L',
    'M&S': 'MKS.L',
    'Aggreko': 'AGK.L',
    'Inovio': 'INO',
    'Berkshire Hathaway Inc.': 'BRK-B',
    'Verizon': 'VZ',
    'Burberry': 'BRBY.L',
    'Citi': 'C',
    'Hastings Group': 'HST.L',
    'Thetrainline.com': 'TRN.L',
    'Morrissons': 'MRW.L',
    'Aberdeen Asset Management': 'ADG.L',
    'Citigroup': 'C',
    'Mylan': 'MYL',
    'Lufthansa': 'LHA',
    'ITV': 'ITV.L',
    'Wilshire Bancorp': 'WLBP',
    'San Miguel': 'SMC.PS',
    'Dialog Semiconductor': 'DLG.L',
    'MillerCoors': 'TAP',
    'Kipa': 'MIGRS.IS',
    'Aer Lingus': 'EIN.L',
    'Mr Bricolage': 'MBB.FP',
    'Taylor Wimpey': 'TW.L',
    'Cadbury': 'MDLZ',
    'Standard Bank': 'SBK.J',
    'Primark': 'ABF.L',
    'United Spirits': 'DGE.L',
    'Integrated Silicon Solution': 'ISSI',
    'Diageo': 'DEO',
    'Hammerson': 'HMSO.L',
}

In [10]:
import yfinance as yf


def is_ticker(symbol):
    return symbol.isupper() and symbol.isalpha() and len(symbol) <= 5


def get_ticker(company):
    # Try to map company name to ticker first
    ticker = company_to_ticker.get(company)
    if ticker:
        return ticker

    # If not found, check if input already looks like a ticker
    if is_ticker(company):
        return company
    return None


# Create a 'Ticker' column by applying the above function
data['Ticker'] = data['Company'].apply(get_ticker)

# Filter valid tickers (drop missing ones)
valid_tickers = data['Ticker'].dropna().unique()

# Lookup full company name and industry from yfinance for valid tickers
ticker_info = {}
for ticker in valid_tickers:
    try:
        t = yf.Ticker(ticker)
        info = t.info
        industry = info.get("industry", "Unknown")
        ticker_info[ticker] = {"Industry": industry}
        print('Fetched info for:', ticker)
    except Exception:
        ticker_info[ticker] = {"Industry": "Unknown"}
        print('Failed to fetch info for:', ticker)

# Convert dict to DataFrame
info_df = pd.DataFrame.from_dict(ticker_info, orient="index").reset_index()
info_df.columns = ["Ticker", "Industry"]

# Merge the fetched info back into your main data
data = data.merge(info_df, how="left", on="Ticker")

Fetched info for: RMG.L
Fetched info for: WEIR.L
Fetched info for: AZN.L
Fetched info for: LLOY.L
Fetched info for: STAN.L
Fetched info for: SVT.L
Fetched info for: GLEN.L
Fetched info for: PRGO
Fetched info for: SHP.L
Fetched info for: BARC.L
Fetched info for: EZJ.L
Fetched info for: CPG.L
Fetched info for: BNZL.L
Fetched info for: AV.L
Fetched info for: DLG.L
Fetched info for: RBS
Fetched info for: MRW.L
Fetched info for: TSCO.L
Fetched info for: SBRY.L
Fetched info for: RDSA.L
Fetched info for: EQNR
Fetched info for: BAER.SW
Fetched info for: DC.L


HTTP Error 404: 


Fetched info for: WLBP
Fetched info for: ITV.L
Fetched info for: CMCSA
Fetched info for: 4502.T
Fetched info for: BP
Fetched info for: NVS
Fetched info for: LGEN.L
Fetched info for: MGGT.L
Fetched info for: KGF.L
Fetched info for: BHP
Fetched info for: AHT.L
Fetched info for: UBI
Fetched info for: SMC.PS
Fetched info for: BXLT
Fetched info for: BAE
Fetched info for: SGE.L
Fetched info for: GNK.L
Fetched info for: ARM
Fetched info for: CAG
Fetched info for: IHG.L
Fetched info for: LSE
Fetched info for: CNA.L
Fetched info for: WG.L
Fetched info for: GSK.L
Fetched info for: RIO
Fetched info for: JPM
Fetched info for: SDR.L
Fetched info for: ICE
Fetched info for: BBY.L
Fetched info for: ITRK.L
Fetched info for: SNY
Fetched info for: LSE.L
Fetched info for: TLW.L
Fetched info for: WOS.L
Fetched info for: PRU.L
Fetched info for: WPP
Fetched info for: 2222.SR
Fetched info for: SAB
Fetched info for: CABK.MC
Fetched info for: GE
Fetched info for: TATASTEEL.NS


HTTP Error 404: 


Fetched info for: OCBC
Fetched info for: BRK-B


...

Fetched info for: HDSI
Fetched info for: INCY
Fetched info for: MKS.L
Fetched info for: GKN
Fetched info for: CIB
Fetched info for: AGK.L
Fetched info for: CRH
Fetched info for: AXA
Fetched info for: BRBY.L
Fetched info for: C


HTTP Error 404: 


Fetched info for: HST.L
Fetched info for: TRN.L
Fetched info for: GSK
Fetched info for: IAG
Fetched info for: MYL
Fetched info for: LHA


HTTP Error 404: 


Fetched info for: CNPC
Fetched info for: AMGN
Fetched info for: EXPE
Fetched info for: ENDP
Fetched info for: CTRP
Fetched info for: CSX
Fetched info for: CAFN
Fetched info for: ORCL
Fetched info for: CELG
Fetched info for: GWW
Fetched info for: GOOGL
Fetched info for: BBBY
Fetched info for: WDC
Fetched info for: WYN
Fetched info for: TRIP
Fetched info for: FXE
Fetched info for: SAP
Fetched info for: CTXS
Fetched info for: DISH
Fetched info for: AAL
Fetched info for: W
Fetched info for: JNPR
Fetched info for: LOW
Fetched info for: JD
Fetched info for: VNET
Fetched info for: SOPH.L
Fetched info for: PFC.L
Fetched info for: HOT
Fetched info for: HIK.L
Fetched info for: BATS.L
Fetched info for: ZURN.SW
Fetched info for: UBS
Fetched info for: BA.L
Fetched info for: FAST
Fetched info for: SYMC
Fetched info for: STX
Fetched info for: TSCO


In [11]:
# Leftover non public companies and their industries
industry_mappings = {
    'Springleaf': 'Consumer Finance',
    'PwC': 'Professional Services',
    "Cadbury's": 'Packaged Foods',
    'Movantik': 'Biotechnology',
    'Sensex': 'Financial Services (Index/Market)',
    'MOVANTIK': 'Biotechnology',
    'MediaCityUK': 'Real Estate',
    'Rangold': 'Precious Metals',
    'Friends': 'Insurance',
    'Standard Bank': 'Banks',
    'Germanwings': 'Airlines',
    'TalkTalk': 'Telecom Services',
    'Imperial Tobacco': 'Tobacco',
    'Sports Direct': 'Specialty Retail',
    'Old Mutual': 'Insurance',
    'BAT': 'Tobacco',
    'Integrated Silicon Solution': 'Semiconductors',
    'Barratt': 'Homebuilding',
    'Lidl': 'Food Retail',
    'Gazprom': 'Oil & Gas - Integrated',
    'China Merchants Group': 'Conglomerate',
    'Aberdeen AM': 'Asset Management',
    'FIO': 'Oil & Gas Exploration & Production',
    'P': 'Consumer Finance',
    'IACI': 'Oil & Gas Services & Equipment',
    'PCLN': 'Internet Content & Information',
    'TZA': 'Financial Services (ETF)',
    'YMI': 'Consumer Finance (ETF)',
    'WFT': 'Oil & Gas Exploration & Production',
    'SPPI': 'Oil & Gas Services & Equipment',
    'SPY': 'Financial Services (ETF)',
    'GLD': 'Financial Services (ETF/Commodity)',
    'ACOM': 'Oil & Gas Exploration & Production',
    'ZAGG': 'Consumer Electronics',
    'MWW': 'Specialty Retail',
    'SDS': 'Financial Services (ETF)',
    'IYT': 'Industrials (ETF)',
    'DUG': 'Oil & Gas Exploration & Production',
    'SLW': 'Precious Metals',
    'HSC': 'Oil & Gas Exploration & Production',
    'GMCR': 'Beverages - Non-Alcoholic',
    'XLF': 'Financial Services (ETF)',
    'VXX': 'Financial Services (ETF)',
    'GPS': 'Specialty Retail',
    'FAZ': 'Financial Services (ETF)',
    'BRCM': 'Semiconductors',
    'ELN': 'Oil & Gas Exploration & Production',
    'TSPT': 'Energy Equipment & Services',
    'SKH': 'Real Estate Investment Trusts (REITs)',
    'DIA': 'Financial Services (ETF)',
    'ZSL': 'Financial Services (ETF)',
    'FMCN': 'Oil & Gas Exploration & Production',
    'MCP': 'Biotechnology',
    'ACE': 'Insurance',
    'DWA': 'Financial Services (ETF)',
    'SINA': 'Internet Content & Information',
    'YHOO': 'Internet Content & Information',
    'CERN': 'Healthcare Technology',
    'RDC': 'Financial Services',
    'HITK': 'Biotechnology',
    'UUP': 'Financial Services (ETF)',
    'MAKO': 'Biotechnology',
    'SSRI': 'Biotechnology',
    'VVUS': 'Biotechnology',
    'CREE': 'Semiconductors',
    'RENN': 'Retail - Apparel & Specialty',
    'ZNGA': 'Interactive Home Entertainment',
    'EFUT': 'Financial Services (ETF)',
    'SWY': 'Food Retail',
    'SKS': 'Restaurants',
    'QCOR': 'Biotechnology',
    'HK': 'Semiconductors',
    'NIHD': 'Biotechnology',
    'KCG': 'Capital Markets',
    'HZNP': 'Biotechnology',
    'AGU': 'Oil & Gas Exploration & Production',
    'ALTR': 'Semiconductors',
    'COG': 'Oil & Gas Exploration & Production',
    'WAC': 'Oil & Gas Exploration & Production',
    'SOX': 'Semiconductors',
    'BONE': 'Biotechnology',
    'SONC': 'Restaurants',
    'KITD': 'Biotechnology',
    'NSM': 'Oil & Gas Exploration & Production',
    'MFLX': 'Biotechnology',
    'ARNA': 'Biotechnology',
    'DMND': 'Specialty Chemicals',
    'JNK': 'Financial Services (ETF)',
    'QQQ': 'Financial Services (ETF)',
    'ISR': 'Biotechnology',
    'FB': 'Internet Content & Information',
    'SAVE': 'Airlines',
    'PKT': 'Biotechnology',
    'BBRY': 'Communication Equipment',
    'ASTX': 'Biotechnology',
    'JE': 'Financial Services',
    'XLB': 'Materials - Diversified Chemicals',
    'TYC': 'Electrical Components & Equipment',
    'QIHU': 'Internet Content & Information',
    'YNDX': 'Internet Content & Information',
    'SHOR': 'Oil & Gas Exploration & Production',
    'UNXL': 'Biotechnology',
    'GOL': 'Airlines',
    'SMH': 'Semiconductors (ETF)',
    'SPX': 'Financial Services (ETF)',
    'OCN': 'Oil & Gas Exploration & Production',
    'UVXY': 'Financial Services (ETF)',
    'SLV': 'Financial Services (ETF/Commodity)',
    'NUGT': 'Financial Services (ETF)',
    'DARA': 'Biotechnology',
    'VOLC': 'Oil & Gas Exploration & Production',
    'YOKU': 'Internet Content & Information',
    'AMCN': 'Biotechnology',
    'CNDO': 'Biotechnology',
    'RNN': 'Oil & Gas Exploration & Production',
    'PRGN': 'Biotechnology',
    'IWM': 'Financial Services (ETF)',
    'COH': 'Retail - Apparel & Specialty',
    'TWTR': 'Internet Content & Information',
    'RAD': 'Specialty Retail',
    'RSOL': 'Oil & Gas Exploration & Production',
    'ARIA': 'Biotechnology',
    'ATHN': 'Biotechnology',
    'KIOR': 'Oil & Gas Exploration & Production',
    'LNKD': 'Internet Content & Information',
    'GTAT': 'Semiconductors',
    'AEGN': 'Biotechnology',
    'UBNT': 'Communication Equipment',
    'MXWL': 'Semiconductors',
    'SODA': 'Beverages - Non-Alcoholic',
    'JNUG': 'Financial Services (ETF)',
    'KORS': 'Retail - Apparel & Specialty',
    'BOBE': 'Restaurants',
    'CHRM': 'Biotechnology',
    'NQ': 'Financial Services (ETF)',
    'RSH': 'Real Estate',
    'SGYP': 'Biotechnology',
    'DBO': 'Financial Services (ETF/Commodity)',
    'WX': 'Biotechnology',
    'AKS': 'Metals & Mining',
    'UGAZ': 'Energy - Oil & Gas',
    'PLNR': 'Semiconductors',
    'YANG': 'Financial Services (ETF)',
    'HYG': 'Financial Services (ETF)',
    'CYTX': 'Biotechnology',
    'WFM': 'Food Retail',
    'DGI': 'Financial Services (ETF)',
    'REN': 'Biotechnology',
    'GALE': 'Biotechnology',
    'RUSS': 'Financial Services (ETF)',
    'ATVI': 'Interactive Home Entertainment',
    'TTT': 'Oil & Gas Exploration & Production',
    'BXS': 'Banks',
    'FRP': 'Financial Services (ETF)',
    'BIOC': 'Biotechnology',
    'TNH': 'Biotechnology',
    'VMW': 'Software',
    'RXII': 'Biotechnology',
    'HCP': 'Real Estate Investment Trusts (REITs)',
    'ONTY': 'Biotechnology',
    'BWLD': 'Restaurants',
    'FREE': 'Biotechnology',
    'NUAN': 'Software',
    'XLE': 'Energy (ETF)',
    'USO': 'Energy (ETF)',
    'IMRS': 'Biotechnology',
    'IBB': 'Biotechnology (ETF)',
    'HDSI': 'Biotechnology',
    'Cairn Homes': 'Homebuilding',
    'Sophos': 'Software',
    'Lehman': None,
    'Blinkbox': 'Entertainment',
    'Actelion': 'Biotechnology',
    'Lemann': None,
    'Towergate': 'Insurance',
    'Starwood': 'Real Estate Investment Trusts (REITs)',
    'Ocwen': 'Mortgage Finance',
    'Nikkei': 'Financial Services (Index)',
    'Coutts': 'Private Banking',
    'CTRP': 'Internet Content & Information',
    'SYMC': 'Software',
    'ENDP': 'Biotechnology',
    'BBBY': 'Specialty Retail',
    'GKN': 'Aerospace & Defense',
    'Gleneagles Hotel': 'Hotels, Resorts & Cruise Lines',
    'Ennismore Group': 'Hotels, Resorts & Cruise Lines',
    'Aggreko': 'Machinery',
    'AXA': 'Insurance',
    'Asda': 'Food Retail',
    'Omnis Pharmaceuticals': 'Biotechnology',
    'Hastings Group': 'Insurance',
    'Horizonte': 'Metals & Mining',
    'Morrissons': 'Food Retail',
    'Aberdeen Asset Managment': 'Asset Management',
    'ARM Holdings': 'Semiconductors',
    'SAB Miller': 'Beverages - Brewers',
    'Mylan': 'Pharmaceuticals',
    'Holcim Lafarge': 'Building Materials',
    'Lufthansa': 'Airlines',
    'CNPC': 'Oil & Gas Integrated',
    'Kinder Morgan': 'Oil & Gas Midstream',
    'ExxonMobil': 'Oil & Gas Integrated',
    'Reed Elsevier': 'Publishing',
    'Samarco': 'Metals & Mining',
    'CAFN': 'Financial Services',
    'CELG': 'Biotechnology',
    'WYN': 'Hotels, Resorts & Cruise Lines',
    'FXE': 'Financial Services (ETF)',
    'CTXS': 'Software',
    'DISH': 'Media',
    'Springleaf': 'Consumer Finance',
    'PwC': 'Professional Services',
    "Cadbury's": 'Packaged Foods',
    'Movantik': 'Biotechnology',
    'Sensex': 'Financial Services (Index/Market)',
    'Royal Mail': 'Integrated Freight & Logistics',
    'ZS Pharma': 'Pharmaceuticals',
    'Valeant': 'Pharmaceuticals',
    'Shire': 'Pharmaceuticals',
    'SABMiller': 'Beverages – Brewers',
    'RBS': 'Banks',
    'Bloomberg': 'Financial Exchanges & Data',
    'Morrisons': 'Food Retail',
    'Shell': 'Oil & Gas - Integrated',
    'Dixons Carphone': 'Specialty Retail',
    'Wilshire Bancorp': 'Regional Banks',
    'Home Retail Group': 'Specialty Retail',
    'Meggitt': 'Aerospace & Defense',
    'Debenhams': 'Specialty Retail',
    'UBI': 'Banks',
    'Home Retail': 'Specialty Retail',
    'San Miguel': 'Packaged Foods',
    'Baxalta': 'Pharmaceuticals',
    'BAE': 'Aerospace & Defense',
    'Astrazeneca': 'Pharmaceuticals',
    'Standard Life': 'Insurance',
    'Greene King': 'Restaurants',
    'BG Group': 'Oil & Gas - Integrated',
    'Lonmin': 'Metals & Mining',
    'Legal & General': 'Insurance',
    'Arm': 'Semiconductors',
    'Rival National Grid plc': 'Electric Utilities',
    'Intertek Group': 'Professional Services',
    'London Stock Exchange': 'Financial Exchanges & Data',
    'Wolseley': 'Construction Materials',
    'Royal Dutch Shell': 'Oil & Gas - Integrated',
    'SAB': 'Beverages – Brewers',
    'Entertainment One': 'Entertainment',
    'Bank BPH': 'Banks',
    'Tower Development': 'Real Estate Development',
    'Lazada': 'Internet Content & Information',
    'Caixabank': 'Banks',
    'OCBC': 'Banks',
    'Berkshire': 'Multi-Sector Holdings',
    'Dialog': 'Semiconductors',
    'Hargreaves Lansdown': 'Capital Markets',
    'AB InBrev': 'Beverages – Brewers',
    'Det Norske': 'Oil & Gas Exploration & Production',
    'Deutsche Börse': 'Financial Exchanges & Data',
    'Kipa': 'Food Retail',
    'Coca-Cola Companp': 'Beverages – Non‑Alcoholic',
    'JV Partner': 'Unknown',
    'Great Portland Estates plc': 'Real Estate Investment Trusts (REITs)',
    'Perssimon': 'Homebuilding',
    'Ryanair': 'Airlines',
    'Lafarge Holcim': 'Building Materials',
    'G4S': 'Commercial Services & Supplies',
    'Friends Life': 'Insurance',
    'Credit Suisse': 'Investment Banking & Brokerage',
    'Aer Lingus': 'Airlines',
    'Insight': 'Professional Services',
    'Mr Bricolage': 'Specialty Retail',
    'Tailor Wimpey': 'Homebuilding',
    'Lehman': 'Investment Banking & Brokerage',
    'Lemann': 'Unknown'
}

In [12]:
# Map non public companies to their industries
def mapping_industry(row):
    return industry_mappings.get(row['Company'], row['Industry'])


data['Industry'] = data.apply(mapping_industry, axis=1)
data.head()

,_id,Sentence,Company,Sentiment,Type,Coarse-Grained Event,Fine-Grained Event,Ticker,Industry
0,1,Royal Mail chairman Donald Brydon set to step ...,Royal Mail,NEG,headline,Management Affairs,"Directors, Supervisors, and Senior Executives ...",RMG.L,Integrated Freight & Logistics
1,100,Slump in Weir leads FTSE down from record high,Weir,NEG,headline,Stock Affairs,Stock Price Movement,WEIR.L,Specialty Industrial Machinery
2,1000,AstraZeneca wins FDA approval for key new lung...,AstraZeneca,POS,headline,Compliance and Credit,Regulatory Inquiries,AZN.L,Drug Manufacturers - General
3,1002,UPDATE 1-Lloyds to cut 945 jobs as part of 3-y...,Lloyds,NEG,headline,Business Operations,Other Business Operations Affairs,LLOY.L,Banks - Regional
4,1005,Standard Chartered Shifts Emerging-Markets Str...,Standard Chartered,NEG,headline,Business Operations,Other Business Operations Affairs,STAN.L,Banks - Diversified


In [13]:
# Drop ticker column
data = data.drop(columns=['Ticker'], axis=1)

In [14]:
# Save cleaned and processed data to a CSV file
data.to_csv('efsa_sentiment_classification.csv', index=False)